# Clean SHMR notebook: joint HOD comparison across DES Deep Fields

This notebook is a stripped-down, analysis-ready version focused on one goal:

**measure and plot the HOD-derived stellar–halo mass relation comparison** for:
- **all fields combined**
- **all fields except SN-E2**
- **COSMOS only**
- **Shuntov +22**

The comparison is done in the redshift bin **0.2 < z < 0.5** and at stellar-mass thresholds  
**log10(M*/M⊙) = 8.5, 9.5, 10.5**.

The code is organized so that:
1. catalogs are loaded and cleaned once per field
2. expensive fit results can be saved to disk with `pickle`
3. plotting can be repeated without rerunning the full pipeline

## Physical idea

The observable is the **angular two-point correlation function** \(w(\theta)\), which measures the excess probability of finding galaxy pairs at angular separation \(\theta\) relative to a random catalog.

A **Halo Occupation Distribution (HOD)** model connects that clustering signal to dark-matter halos through three parameters in the simplified model used here:
- **log Mmin**: minimum halo mass to host a central galaxy
- **log M1**: halo mass where satellites become common
- **alpha**: slope of the satellite occupation

For a **stellar-mass threshold** sample, the fitted **log Mmin** is used here as the characteristic halo-mass point for the SHMR comparison.

Why compare multiple field combinations?
- The thesis shows that the four DES deep fields are affected by **cosmic variance**, and that **SN-E2 can appear overdense in some bins**, which can shift the inferred halo masses.
- Combining independent fields is therefore more robust than relying on a single line of sight.
- Plotting **all fields**, **without SN-E2**, and **COSMOS only** lets you directly see how sensitive the SHMR is to field selection.

Why compare with **Shuntov +22**?
- It is an external COSMOS-based reference relation, so it acts as a useful consistency check for the halo-mass scale.

In [1]:
%matplotlib inline

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

from astropy.table import Table, join, vstack
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
import DarkVerse as dv


/home/astro/manasoi7/.conda/envs/project/lib/python3.12/site-packages/halomod/halo_model.py:32: UserWarning: Warning: Some Halo-Exclusion models have significant speedup when using Numba
  from .halo_exclusion import NoExclusion


In [2]:

plt.rc('font', family='serif', size=14)

## Configuration

In [3]:
CONFIG = {
    'min_sep': 0.003,
    'max_sep': 1.78,
    'bin_size': 0.1,
    'sep_units': 'degrees',
    'var_method': 'shot'
}

Z_RANGE = (0.2, 0.5)

# Threshold samples: each bin is interpreted as M* > SM_min
SM_BINS = [
    (8.5, 12.5),
    (9.5, 12.5),
    (10.5, 12.5),
]

home_dir = os.path.expanduser('~')
DATA_PATH = os.path.join(home_dir, 'Master_Thesis', 'DATA')
RESULTS_DIR = os.path.join(home_dir, 'Master_Thesis', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

FIELD_COLORS = {
    'SN-X3': 'tab:blue',
    'SN-C3': 'tab:orange',
    'SN-E2': 'tab:green',
    'COSMOS': 'tab:red',
}

In [4]:
ALL_FIELDS = {
    "SN-X3": {
        "catalog": "DES_DF_SN-X3_sbi_output.fits",
        "masked": "SN-X3_masked_cat.fits",
        "randoms": "SN-X3_randoms_ugriz_trim_video.fits",
    },
    "SN-C3": {
        "catalog": "DES_DF_SN-C3_sbi_output.fits",
        "masked": "SN-C3_masked_cat.fits",
        "randoms": "SN-C3_randoms_ugriz_trim_video.fits",
    },
    "SN-E2": {
        "catalog": "DES_DF_SN-E2_sbi_output.fits",
        "masked": "SN-E2_masked_cat.fits",
        "randoms": "SN-E2_randoms_ugriz_trim_video.fits",
    },
    "COSMOS": {
        "catalog": "DES_DF_COSMOS_sbi_output.fits",
        "masked": "COSMOS_masked_cat.fits",
        "randoms": "COSMOS_randoms_ugriz_trim_video.fits",
    },
}

FIELDS_NO_SNE2 = {k: v for k, v in ALL_FIELDS.items() if k != "SN-E2"}
COSMOS_ONLY = {"COSMOS": ALL_FIELDS["COSMOS"]}

## Catalog cleaning and in-memory caching

These helpers do two different jobs:

- **cleaning**: standardize columns, remove stars, remove flagged objects
- **caching**: keep the already-loaded tables in memory during the current kernel session so repeated calls are fast

In [5]:
def load_catalog(path):
    cat = Table.read(path)

    # Normalize column names to lowercase once
    old_cols = list(cat.colnames)
    new_cols = [c.lower() for c in old_cols]
    if old_cols != new_cols:
        cat.rename_columns(old_cols, new_cols)

    return cat


def remove_stars(cat, compactness_cut=-0.02):
    # kNN star/galaxy classification
    if 'knn_class' in cat.colnames:
        cat = cat[cat['knn_class'] != 2]

    # Morphology-based compactness cut: galaxies tend to have bdf-psf < threshold
    bdf = 'bdf_mag_dered_calib_i'
    psf = 'psf_mag_dered_calib_i'
    if bdf in cat.colnames and psf in cat.colnames:
        delta = cat[bdf] - cat[psf]
        cat = cat[delta < compactness_cut]

    return cat


def remove_flagged(cat):
    mask = np.ones(len(cat), dtype=bool)
    if 'flags' in cat.colnames:
        mask &= (cat['flags'] == 0)
    if 'flags_nir' in cat.colnames:
        mask &= (cat['flags_nir'] == 0)
    return cat[mask]


def standardize_catalog(cat):
    cat = cat.copy()
    rename_map = {
        'mode_z': 'z',
        'mode_mass': 'sm',
        'ra_1': 'ra',
        'dec_1': 'dec',
        'sm': 'SM',
        'z': 'z',
    }

    # Only rename when the source exists and destination does not
    for old, new in list(rename_map.items()):
        if old in cat.colnames and new not in cat.colnames:
            cat.rename_column(old, new)

    return cat


def clean_catalog(path):
    cat = load_catalog(path)
    cat = remove_stars(cat, compactness_cut=-0.02)
    cat = remove_flagged(cat)
    cat = standardize_catalog(cat)
    return cat


_catalog_cache = {}
_random_cache = {}


def get_clean_joined_catalog(field_name, field_paths):
    if field_name in _catalog_cache:
        return _catalog_cache[field_name].copy()

    cat = clean_catalog(os.path.join(DATA_PATH, field_paths["catalog"]))
    masked = clean_catalog(os.path.join(DATA_PATH, field_paths["masked"]))
    joined = join(cat, masked, keys='id')
    joined = standardize_catalog(joined)

    _catalog_cache[field_name] = joined
    return joined.copy()


def get_clean_randoms(field_name, field_paths):
    if field_name in _random_cache:
        return _random_cache[field_name].copy()

    rand = clean_catalog(os.path.join(DATA_PATH, field_paths["randoms"]))
    rand = standardize_catalog(rand)

    _random_cache[field_name] = rand
    return rand.copy()

## Building threshold selections

Each selection creates a threshold galaxy sample in one field and one redshift range, computes \(w(\theta)\), and stores the objects needed for HOD fitting.

In [6]:
def build_selection(field_name, field_paths, z_min, z_max, sm_min, sm_max, config):
    cat = get_clean_joined_catalog(field_name, field_paths)
    rand = get_clean_randoms(field_name, field_paths)

    sub = dv.Selection(
        cat,
        rand,
        z_min,
        z_max,
        sm_min,
        sm_max,
        config
    )

    # Attach the field name for plotting and bookkeeping
    sub.field_name = field_name
    return sub


def interpolate_selection_to_theta(selection, shared_theta):
    interp_w = interp1d(
        selection.theta,
        selection.w_theta,
        kind='linear',
        bounds_error=False,
        fill_value='extrapolate'
    )
    interp_var = interp1d(
        selection.theta,
        selection.var_w_theta_bootstrap,
        kind='linear',
        bounds_error=False,
        fill_value='extrapolate'
    )

    selection.w_theta = interp_w(shared_theta)
    selection.var_w_theta_bootstrap = interp_var(shared_theta)
    selection.theta = np.asarray(shared_theta)
    return selection

## Joint HOD fitting

The idea is simple: fit one common set of HOD parameters to several independent fields at once.

This is useful because the underlying galaxy–halo connection should be much more stable than the field-to-field fluctuations caused by cosmic variance.

In [7]:
class MultiFieldHODFitter:
    def __init__(self, selections, fit_mask=(0.004, 0.4)):
        self.selections = selections
        self.fit_mask = fit_mask

        theta0 = np.asarray(selections[0].theta)
        for sel in selections[1:]:
            assert np.allclose(sel.theta, theta0), "All selections must share the same theta grid"

        self.theta = theta0
        self.mask = (self.theta >= fit_mask[0]) & (self.theta <= fit_mask[1])

    def _combined_model(self, theta_fit, logM_min_scaled, logM_1, alpha):
        model_parts = []
        for sel in self.selections:
            xi_g = sel.hod_model(logM_min_scaled * 1e7, logM_1, alpha)
            model_parts.append(np.asarray(xi_g)[self.mask])
        return np.concatenate(model_parts)

    def _combined_data(self):
        w_all, w_err_all = [], []
        for sel in self.selections:
            w_all.append(np.asarray(sel.w_theta)[self.mask])
            w_err_all.append(np.sqrt(np.asarray(sel.var_w_theta_bootstrap))[self.mask])
        return np.concatenate(w_all), np.concatenate(w_err_all)

    def fit(self, p0=None, bounds=None):
        if p0 is None:
            p0 = [12.59 * 1e-7, 13.97, 1.0]

        if bounds is None:
            # alpha is fixed to 1 here to match the threshold-comparison setup
            bounds = ([11.0 * 1e-7, 13.0, 0.99], [14.5 * 1e-7, 15.5, 1.01])

        w_obs, w_err = self._combined_data()

        popt, pcov = curve_fit(
            self._combined_model,
            self.theta[self.mask],
            w_obs,
            sigma=w_err,
            p0=p0,
            bounds=bounds,
            maxfev=20000
        )

        # Save the best-fit model back onto each selection for later plotting
        for sel in self.selections:
            sel.hod_params = popt
            sel.xi_g = sel.hod_model(popt[0] * 1e7, popt[1], popt[2])

        return popt, pcov

In [8]:
def plot_joint_fit(field_selections, colors, sm_min, sm_max):
    plt.figure(figsize=(7, 5))

    for sel in field_selections:
        field_name = getattr(sel, 'field_name', 'Field')
        theta = np.asarray(sel.theta)
        w_theta = np.asarray(sel.w_theta)
        yerr = np.sqrt(np.asarray(sel.var_w_theta_bootstrap))
        color = colors.get(field_name, None)

        plt.errorbar(
            theta, w_theta, yerr=yerr,
            fmt='o', ms=4, capsize=2,
            label=field_name, color=color, alpha=0.9
        )

        if hasattr(sel, 'xi_g') and sel.xi_g is not None:
            plt.plot(theta, np.asarray(sel.xi_g), '-', lw=1.8, color=color)

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel(r'$\theta$ [deg]')
    plt.ylabel(r'$w(\theta)$')
    plt.title(f'Joint HOD fit: log10(M*/M⊙) > {sm_min:.1f}')
    plt.grid(True, which='both', alpha=0.25)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()

In [9]:
def run_joint_hod_analysis(fields, sm_bins, z_range, config, colors, plot_each_bin=False):
    z_min, z_max = z_range
    results_by_bin = {}

    for sm_min, sm_max in sm_bins:
        print(f"Processing threshold log10(M*/M⊙) > {sm_min:.1f}")

        # Build first field to define the common theta grid
        first_field_name = next(iter(fields))
        first_selection = build_selection(
            first_field_name,
            fields[first_field_name],
            z_min,
            z_max,
            sm_min,
            sm_max,
            config
        )
        shared_theta = np.asarray(first_selection.theta)

        field_selections = []
        bias_eff_dict = {}

        for field_name, field_paths in fields.items():
            sub = build_selection(field_name, field_paths, z_min, z_max, sm_min, sm_max, config)
            sub = interpolate_selection_to_theta(sub, shared_theta)

            field_selections.append(sub)
            if hasattr(sub, 'gg') and hasattr(sub.gg, 'bias_effective_tracer'):
                bias_eff_dict[field_name] = sub.gg.bias_effective_tracer
            else:
                bias_eff_dict[field_name] = np.nan

        fitter = MultiFieldHODFitter(field_selections)
        hod_params, pcov = fitter.fit()

        results_by_bin[(sm_min, sm_max)] = {
            'field_selections': field_selections,
            'hod_params': hod_params,
            'pcov': pcov,
            'bias_eff': bias_eff_dict,
            'fields_used': list(fields.keys()),
        }

        print(
            f"  best fit: logMmin={hod_params[0] * 1e7:.3f}, "
            f"logM1={hod_params[1]:.3f}, alpha={hod_params[2]:.3f}"
        )

        if plot_each_bin:
            plot_joint_fit(field_selections, colors, sm_min, sm_max)

    return results_by_bin

## Save/load helpers for disk persistence

This is separate from in-memory caching.

- **In-memory cache**: fast within the current kernel session only
- **Pickled results**: survive notebook restarts and can be reloaded later

In [10]:
def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def maybe_run_or_load(label, fields, sm_bins, z_range, config, colors, filename, rerun=False, plot_each_bin=False):
    path = os.path.join(RESULTS_DIR, filename)

    if (not rerun) and os.path.exists(path):
        print(f"Loading saved results for {label}: {path}")
        return load_pickle(path)

    print(f"Running analysis for {label}")
    results = run_joint_hod_analysis(
        fields=fields,
        sm_bins=sm_bins,
        z_range=z_range,
        config=config,
        colors=colors,
        plot_each_bin=plot_each_bin
    )
    save_pickle(results, path)
    print(f"Saved results for {label}: {path}")
    return results

## SHMR point extraction

For the threshold samples, we take the fitted **log Mmin** as the halo-mass point associated with the stellar-mass threshold.  
For your own measurements, the uncertainty is therefore shown on the **x-axis**.

For **Shuntov +22**, you asked to keep the uncertainty on the **y-axis**, using the 16th and 84th percentile spread in stellar mass around the matched reference points.

In [11]:
def extract_threshold_mass_points(results_by_bin):
    rows = []

    for (sm_min, sm_max), result in results_by_bin.items():
        hod_params = result.get('hod_params')
        pcov = result.get('pcov')

        if hod_params is None or pcov is None:
            continue

        logM_min = hod_params[0] * 1e7
        logM_min_err = np.sqrt(pcov[0, 0]) * 1e7 if np.isfinite(pcov[0, 0]) else np.nan

        rows.append({
            'SM_min': sm_min,
            'SM_max': sm_max,
            'stellar_threshold': sm_min,
            'logM_halo': logM_min,
            'logM_halo_err': logM_min_err,
            'fields_used': ', '.join(result.get('fields_used', [])),
        })

    rows = sorted(rows, key=lambda r: r['stellar_threshold'])
    return rows

In [12]:
def build_shuntov22_reference():
    thresholds = np.array([8.5, 9.5, 10.5], dtype=float)

    Mhalo = np.array([
        66296706817.30795, 73917898798.73352, 82415191117.65851, 91889296602.63226, 102452505607.51738,
        114230016915.3509, 127361421637.34383, 142002357694.69983, 158326354492.74042, 176526889651.04785,
        196819682166.6986, 219445249189.9458, 244671756716.13455, 272798197981.23068, 304157937232.3985,
        339122660875.61456, 378106782830.6767, 421572356307.3892, 470034550218.98444, 524067755141.1132,
        584312391186.3297, 651482498483.3376, 726374200225.966, 809875138598.8401, 902974995417.5806,
        1006777222177.7738, 1122512118541.2837, 1251551414270.6223, 1395424527441.4702, 1555836691631.2769,
        1734689166933.6006, 1934101774346.0808, 2156438020617.3528, 2404333111341.2197, 2680725184318.9062,
        2988890133377.9307, 3332480435391.469, 3715568440689.3813, 4142694639953.733, 4618921479674.62,
        5149893364005.711, 5741903554181.688, 6401968758413.307, 7137912296323.953, 7958456823622.604,
        8873327716016.268, 9893368337705.371, 11030668560661.88, 12298708057947.068, 13712516069428.059,
        15288849533494.11, 17046391696047.846, 19005973550756.92, 21190820735148.2, 2404333111341.2197
    ], dtype=float)

    Mcent_50 = np.array([
        138185889.15720302, 172720900.0668386, 215673222.45576337, 269089919.88765955, 335515179.327528,
        418111484.28001314, 520828269.08945644, 648606120.6643087, 807610719.2412863, 1005539328.3203484,
        1252045100.3530602, 1559341768.9483867, 1942821473.3047915, 2421887213.518848, 3020837737.8737206,
        3768184888.123724, 4697571006.331157, 5849459111.900749, 7272065288.372202, 9012562604.540678,
        11103654734.995914, 13563337202.82329, 16383512911.519424, 19526092461.489563, 22964843744.2323,
        26653770811.350174, 30497339591.234303, 34332934621.112762, 38039137854.068245, 41580202752.4853,
        44960629691.55288, 48255164000.67649, 51559726515.37951, 54848574060.27377, 58074904513.471115,
        61195354453.304756, 64176275624.11628, 66982124975.57955, 69594247688.89069, 72041324787.95726,
        74431053359.4038, 76855424127.0457, 79317242371.72003, 81830017335.21799, 84437227153.89256,
        87197196144.7057, 90094865412.7629, 93078040073.30054, 96062580938.40805, 98928121262.40611,
        101605473415.01775, 104098955603.05504, 106479288961.97887, 108926724434.02715, 131113681632.74998
    ], dtype=float)

    Mcent_16 = np.array([
        122859549.08492075, 153463093.1277725, 191768882.6401075, 239526223.76712334, 298836177.16694885,
        372219547.2840017, 463052304.6707711, 575984089.142209, 717040702.2843549, 894038906.9945077,
        1116606899.1893206, 1395243177.6998365, 1742212577.8122094, 2171990261.733003, 2701881404.0431743,
        3356430344.5712314, 4169195559.5823936, 5183738595.810069, 6456409992.04091, 8037261534.242731,
        9941610303.771763, 12155006203.772999, 14615543931.667683, 17212255517.138123, 19873379026.627617,
        22520938757.34364, 25036248827.990265, 27250495293.425167, 29045929842.63774, 30407827062.838898,
        31407366014.24033, 32259098525.92094, 33224153465.373096, 34332008079.68556, 35556722949.82484,
        36839738462.00549, 38087147941.07143, 39256742347.15974, 40349198016.17066, 41392236424.72836,
        42456861702.30304, 43598950121.40022, 44811201739.4774, 46072871679.93738, 47348448049.77223,
        48595493415.716606, 49803988231.3955, 50986747058.85586, 52185980770.76462, 53483597074.1377,
        54927179481.6141, 56525638200.18852, 58283289227.600975, 60197536524.2768, 75233860276.46657
    ], dtype=float)

    Mcent_84 = np.array([
        151841683.43116438, 188804430.35792825, 234528445.9761902, 291627479.4423529, 363603417.5064963,
        455136384.68409896, 571674824.6189399, 719219189.8050666, 905012812.7288473, 1137794075.4261613,
        1428466573.0027277, 1791676007.004955, 2246154632.081757, 2815655554.5198855, 3529882573.2354293,
        4421325003.19453, 5525535085.792659, 6882740897.502729, 8537251211.159791, 10529464663.453302,
        12883561796.396748, 15602388043.97047, 18652262648.43852, 21963348732.300957, 25508669710.572273,
        29268192338.128353, 33203775236.438942, 37254202425.78185, 41370327876.53652, 45531477775.037155,
        49735616340.65336, 54015845050.07054, 58416390418.72675, 62921318405.1627, 67510561839.025604,
        72179252980.06393, 76942390560.4907, 81778926561.5444, 86648244541.48576, 91518589845.90186,
        96379988758.53552, 101225557692.34584, 106043833293.65878, 110866749981.52075, 115806777285.15465,
        121023457392.73651, 126511325700.39758, 132197429787.67732, 137965749705.99323, 143650476583.49664,
        149129962573.3049, 154348609108.23242, 159305987750.48944, 164101459416.9993, 195683907020.22842
    ], dtype=float)

    # Flatten and convert to log10
    Mhalo = np.array(Mhalo).flatten()
    Mcent_50 = np.array(Mcent_50).flatten()
    Mcent_16 = np.array(Mcent_16).flatten()
    Mcent_84 = np.array(Mcent_84).flatten()

    log_Mhalo = np.log10(Mhalo)
    log_Mcent_50 = np.log10(Mcent_50)
    log_Mcent_16 = np.log10(Mcent_16)
    log_Mcent_84 = np.log10(Mcent_84)

    # Vertical uncertainties in stellar mass
    y_err_lower = log_Mcent_50 - log_Mcent_16
    y_err_upper = log_Mcent_84 - log_Mcent_50
    y_err = np.array([y_err_lower, y_err_upper])

    # Select the nearest points to the requested thresholds
    indices = [np.argmin(np.abs(log_Mcent_50 - t)) for t in thresholds]

    x_points = log_Mhalo[indices]
    y_points = log_Mcent_50[indices]
    y_err_points = y_err[:, indices]

    rows = []
    for i in range(len(thresholds)):
        rows.append({
            'target_threshold': float(thresholds[i]),
            'stellar_threshold': float(y_points[i]),
            'logM_halo': float(x_points[i]),
            'stellar_err_lower': float(y_err_points[0, i]),
            'stellar_err_upper': float(y_err_points[1, i]),
        })

    return rows

In [13]:
def plot_shmr_threshold_comparison(all_fields_rows, no_sne2_rows, cosmos_rows, shuntov_rows):
    fig, ax = plt.subplots(figsize=(10, 7))

    # Shuntov +22: y-errors
    if shuntov_rows:
        ax.errorbar(
            [r['logM_halo'] for r in shuntov_rows],
            [r['stellar_threshold'] for r in shuntov_rows],
            yerr=[
                [r['stellar_err_lower'] for r in shuntov_rows],
                [r['stellar_err_upper'] for r in shuntov_rows],
            ],
            fmt='o',
            ms=8,
            capsize=5,
            label='Shuntov +22'
        )

    # This work: all fields
    if all_fields_rows:
        ax.errorbar(
            [r['logM_halo'] for r in all_fields_rows],
            [r['stellar_threshold'] for r in all_fields_rows],
            xerr=[r['logM_halo_err'] for r in all_fields_rows],
            fmt='o-',
            ms=8,
            capsize=5,
            label='This work: all fields'
        )

    # This work: without SN-E2
    if no_sne2_rows:
        ax.errorbar(
            [r['logM_halo'] for r in no_sne2_rows],
            [r['stellar_threshold'] for r in no_sne2_rows],
            xerr=[r['logM_halo_err'] for r in no_sne2_rows],
            fmt='s--',
            ms=7,
            capsize=5,
            label='This work: without SN-E2'
        )

    # This work: COSMOS only
    if cosmos_rows:
        ax.errorbar(
            [r['logM_halo'] for r in cosmos_rows],
            [r['stellar_threshold'] for r in cosmos_rows],
            xerr=[r['logM_halo_err'] for r in cosmos_rows],
            fmt='d-.',
            ms=7,
            capsize=5,
            label='This work: COSMOS only'
        )

    ax.set_xlabel(r'$\log_{10}(M_h/M_\odot)$')
    ax.set_ylabel(r'$\log_{10}(M_*/M_\odot)$ threshold')
    ax.set_title('HOD-derived halo mass vs. stellar-mass threshold')
    ax.grid(True, ls='--', alpha=0.4)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [14]:
def threshold_rows_to_table(rows, label):
    table_rows = []
    for r in rows:
        table_rows.append((
            label,
            r.get('stellar_threshold', np.nan),
            r.get('logM_halo', np.nan),
            r.get('logM_halo_err', np.nan),
            r.get('fields_used', ''),
        ))
    return Table(
        rows=table_rows,
        names=['run', 'stellar_threshold', 'logM_halo', 'logM_halo_err', 'fields_used']
    )

## Main run cell

Set `RERUN = False` to reuse pickled results if they already exist.  
Set `RERUN = True` if you changed the code or want to recompute the fits.

In [ ]:
RERUN = False
PLOT_EACH_BIN = False

results_all_fields = maybe_run_or_load(
    label='all fields',
    fields=ALL_FIELDS,
    sm_bins=SM_BINS,
    z_range=Z_RANGE,
    config=CONFIG,
    colors=FIELD_COLORS,
    filename='shmr_all_fields_z02_05.pkl',
    rerun=RERUN,
    plot_each_bin=PLOT_EACH_BIN
)

results_no_sne2 = maybe_run_or_load(
    label='without SN-E2',
    fields=FIELDS_NO_SNE2,
    sm_bins=SM_BINS,
    z_range=Z_RANGE,
    config=CONFIG,
    colors=FIELD_COLORS,
    filename='shmr_no_sne2_z02_05.pkl',
    rerun=RERUN,
    plot_each_bin=PLOT_EACH_BIN
)

results_cosmos = maybe_run_or_load(
    label='COSMOS only',
    fields=COSMOS_ONLY,
    sm_bins=SM_BINS,
    z_range=Z_RANGE,
    config=CONFIG,
    colors=FIELD_COLORS,
    filename='shmr_cosmos_only_z02_05.pkl',
    rerun=RERUN,
    plot_each_bin=PLOT_EACH_BIN
)

Running analysis for all fields
Processing threshold log10(M*/M⊙) > 8.5


/home/astro/manasoi7/.conda/envs/project/lib/python3.12/site-packages/hmf/density_field/transfer_models.py:233: UserWarning: 'extrapolate_with_eh' was not set. Defaulting to True, which is different behaviour than versions <=3.4.4. This warning may be removed in v4.0. Silence it by setting extrapolate_with_eh explicitly.
  warnings.warn(
/home/astro/manasoi7/.conda/envs/project/lib/python3.12/site-packages/halomod/halo_model.py:784: UserWarning: You are using an un-normalized mass function and bias function pair.Bias Tinker10 has the following paired HMF model: (). Matter correlations are not well-defined.
  tools.norm_warn(self)


## Build the SHMR comparison points

In [ ]:
all_fields_rows = extract_threshold_mass_points(results_all_fields)
no_sne2_rows = extract_threshold_mass_points(results_no_sne2)
cosmos_rows = extract_threshold_mass_points(results_cosmos)
shuntov_rows = build_shuntov22_reference()

summary_tables = []
if all_fields_rows:
    summary_tables.append(threshold_rows_to_table(all_fields_rows, 'all fields'))
if no_sne2_rows:
    summary_tables.append(threshold_rows_to_table(no_sne2_rows, 'without SN-E2'))
if cosmos_rows:
    summary_tables.append(threshold_rows_to_table(cosmos_rows, 'COSMOS only'))

if summary_tables:
    summary_table = vstack(summary_tables)
    summary_table

## Final figure

In [ ]:
plot_shmr_threshold_comparison(
    all_fields_rows=all_fields_rows,
    no_sne2_rows=no_sne2_rows,
    cosmos_rows=cosmos_rows,
    shuntov_rows=shuntov_rows
)

## Interpretation checklist

When you inspect the final figure, the main questions are:

1. **Does the all-fields curve look smoother than COSMOS-only?**
   - It should, because combining independent fields reduces sensitivity to cosmic variance.

2. **Does removing SN-E2 move the halo-mass points closer to the external reference?**
   - If yes, that supports the idea that SN-E2 is biasing the joint result in this redshift bin.

3. **Does COSMOS-only track Shuntov +22 better than the DES multi-field result?**
   - That can happen because Shuntov +22 is also COSMOS-based, so field-to-field structure may line up more closely.

4. **Are the shifts larger than the x-error bars?**
   - If so, field selection is not just a noise-level effect; it is physically important for the inferred SHMR.

Because this notebook uses threshold samples, the plotted halo-mass points correspond to the threshold-selected HOD fit rather than a full continuous SHMR reconstruction.